## IEOR4004 Project II — Q1 & Q2 (Scheduling the NBA)

This notebook:
- computes Q1 (a)–(d) from `games.csv`
- builds/solves the Q2 feasibility integer program (no meaningful objective)
- exports a feasible schedule to CSV (`Date, Home, Visitor`)

In [13]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import pandas as pd

# -------------------------
# Data loading + Q1 helpers
# -------------------------


@dataclass
class Q1Data:
    teams: list[str]
    dates: list[str]
    home_on: dict[tuple[str, str], int]
    away_on: dict[tuple[str, str], int]
    home_vs: dict[tuple[str, str], int]
    away_vs: dict[tuple[str, str], int]
    home_dates: dict[str, list[str]]
    away_dates: dict[str, list[str]]


def load_games(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    expected_cols = {"Date", "Visitor", "Home", "Attend.", "Arena", "Notes"}
    missing = expected_cols.difference(df.columns)
    if missing:
        raise ValueError(f"Missing expected columns in games.csv: {sorted(missing)}")

    df["Date_dt"] = pd.to_datetime(df["Date"], format="%a, %b %d, %Y")
    df["Date_label"] = df["Date_dt"].dt.strftime("%a, %b %d, %Y")

    df["Attendance"] = (
        df["Attend."]
        .astype(str)
        .str.replace(",", "", regex=False)
        .replace({"": None, "nan": None})
        .astype("float")
    )

    return df


def compute_q1_data(df: pd.DataFrame) -> Q1Data:
    teams = sorted(set(df["Home"]).union(set(df["Visitor"])))
    dates = sorted(
        df["Date_label"].unique(),
        key=lambda x: pd.to_datetime(x, format="%a, %b %d, %Y"),
    )

    home_dates = {
        t: sorted(
            df.loc[df["Home"] == t, "Date_label"].unique(),
            key=lambda x: pd.to_datetime(x, format="%a, %b %d, %Y"),
        )
        for t in teams
    }
    away_dates = {
        t: sorted(
            df.loc[df["Visitor"] == t, "Date_label"].unique(),
            key=lambda x: pd.to_datetime(x, format="%a, %b %d, %Y"),
        )
        for t in teams
    }

    home_on = {(i, d): int(d in set(home_dates[i])) for i in teams for d in dates}
    away_on = {(i, d): int(d in set(away_dates[i])) for i in teams for d in dates}

    home_counts_df = (
        pd.crosstab(df["Home"], df["Visitor"])\
        .reindex(index=teams, columns=teams, fill_value=0)
    )
    away_counts_df = (
        pd.crosstab(df["Visitor"], df["Home"])\
        .reindex(index=teams, columns=teams, fill_value=0)
    )

    home_vs = {(i, j): int(home_counts_df.loc[i, j]) for i in teams for j in teams if i != j}
    away_vs = {(i, j): int(away_counts_df.loc[i, j]) for i in teams for j in teams if i != j}

    return Q1Data(
        teams=teams,
        dates=dates,
        home_on=home_on,
        away_on=away_on,
        home_vs=home_vs,
        away_vs=away_vs,
        home_dates=home_dates,
        away_dates=away_dates,
    )


def export_q1_outputs(q1: Q1Data, out_dir: Path) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)

    teams = q1.teams

    home_matrix = pd.DataFrame(0, index=teams, columns=teams, dtype=int)
    away_matrix = pd.DataFrame(0, index=teams, columns=teams, dtype=int)
    for i in teams:
        for j in teams:
            if i == j:
                continue
            home_matrix.loc[i, j] = q1.home_vs[(i, j)]
            away_matrix.loc[i, j] = q1.away_vs[(i, j)]

    home_matrix.to_csv(out_dir / "q1_home_vs_counts_matrix.csv", index=True)
    away_matrix.to_csv(out_dir / "q1_away_vs_counts_matrix.csv", index=True)

    long_rows = []
    for i in teams:
        for j in teams:
            if i == j:
                continue
            long_rows.append(
                {
                    "team_i": i,
                    "team_j": j,
                    "home_vs_j_count": q1.home_vs[(i, j)],
                    "away_at_j_count": q1.away_vs[(i, j)],
                }
            )
    pd.DataFrame(long_rows).to_csv(out_dir / "q1_pair_counts_long.csv", index=False)

    home_dates_rows = [{"team": i, "date": d} for i in teams for d in q1.home_dates[i]]
    away_dates_rows = [{"team": i, "date": d} for i in teams for d in q1.away_dates[i]]
    pd.DataFrame(home_dates_rows).to_csv(out_dir / "q1_home_dates.csv", index=False)
    pd.DataFrame(away_dates_rows).to_csv(out_dir / "q1_away_dates.csv", index=False)


# ---------
# File paths
# ---------
# Teammates may run this notebook from different working directories.
# We auto-locate the repo-relative file: Opt Models/Assignment/Project_2/games.csv

_repo_rel_games = Path("Opt Models") / "Assignment" / "Project_2" / "games.csv"

GAMES_CSV = None
for base in [Path.cwd(), *Path.cwd().parents]:
    cand = base / _repo_rel_games
    if cand.exists():
        GAMES_CSV = cand
        break

if GAMES_CSV is None:
    # Fall back: maybe they are already in the Project_2 directory
    cand = Path.cwd() / "games.csv"
    if cand.exists():
        GAMES_CSV = cand

if GAMES_CSV is None:
    raise FileNotFoundError(
        "Could not find games.csv. Tried searching for: "
        f"{_repo_rel_games} up the directory tree, and also ./games.csv. "
        "Please set GAMES_CSV manually."
    )

PROJECT2_DIR = GAMES_CSV.parent
OUT_DIR = PROJECT2_DIR / "outputs_q1_q2"


df = load_games(GAMES_CSV)
q1 = compute_q1_data(df)

len(q1.teams), len(q1.dates)

(16, 16)

In [14]:
# What does `q1` look like?
# `q1` is a Q1Data dataclass (not a single DataFrame). Here are DataFrame views.

q1  # shows fields in the dataclass

Q1Data(teams=['Atlanta Hawks', 'Boston Celtics', 'Brooklyn Nets', 'Chicago Bulls', 'Cleveland Cavaliers', 'Dallas Mavericks', 'Denver Nuggets', 'Golden State Warriors', 'Houston Rockets', 'Los Angeles Lakers', 'Miami Heat', 'Milwaukee Bucks', 'New York Knicks', 'Philadelphia 76ers', 'Phoenix Suns', 'Toronto Raptors'], dates=['Sat, Nov 01, 2025', 'Mon, Nov 03, 2025', 'Wed, Nov 05, 2025', 'Fri, Nov 07, 2025', 'Tue, Nov 11, 2025', 'Thu, Nov 13, 2025', 'Sat, Nov 15, 2025', 'Mon, Nov 17, 2025', 'Wed, Nov 19, 2025', 'Fri, Nov 21, 2025', 'Sun, Nov 23, 2025', 'Thu, Nov 27, 2025', 'Fri, Nov 28, 2025', 'Sat, Nov 29, 2025', 'Mon, Dec 01, 2025', 'Thu, Dec 25, 2025'], home_on={('Atlanta Hawks', 'Sat, Nov 01, 2025'): 0, ('Atlanta Hawks', 'Mon, Nov 03, 2025'): 1, ('Atlanta Hawks', 'Wed, Nov 05, 2025'): 0, ('Atlanta Hawks', 'Fri, Nov 07, 2025'): 1, ('Atlanta Hawks', 'Tue, Nov 11, 2025'): 0, ('Atlanta Hawks', 'Thu, Nov 13, 2025'): 0, ('Atlanta Hawks', 'Sat, Nov 15, 2025'): 1, ('Atlanta Hawks', 'Mon, No

In [15]:
# Long-format (i,j) table 

pair_long_df = pd.DataFrame(
    [
        {
            "team_i": i,
            "team_j": j,
            "home_vs_j_count": q1.home_vs[(i, j)],
            "away_at_j_count": q1.away_vs[(i, j)],
        }
        for i in q1.teams
        for j in q1.teams
        if i != j
    ]
)

pair_long_df.head(20)

,team_i,team_j,home_vs_j_count,away_at_j_count
0,Atlanta Hawks,Boston Celtics,1,0
1,Atlanta Hawks,Brooklyn Nets,0,1
2,Atlanta Hawks,Chicago Bulls,1,1
3,Atlanta Hawks,Cleveland Cavaliers,0,1
4,Atlanta Hawks,Dallas Mavericks,1,0
5,Atlanta Hawks,Denver Nuggets,0,1
6,Atlanta Hawks,Golden State Warriors,1,0
7,Atlanta Hawks,Houston Rockets,0,1
8,Atlanta Hawks,Los Angeles Lakers,1,0
9,Atlanta Hawks,Miami Heat,1,0


In [16]:
# Wide matrices (same info as pair_long_df but in matrix form)

home_vs_matrix = pd.DataFrame(0, index=q1.teams, columns=q1.teams, dtype=int)
away_at_matrix = pd.DataFrame(0, index=q1.teams, columns=q1.teams, dtype=int)

for i in q1.teams:
    for j in q1.teams:
        if i == j:
            continue
        home_vs_matrix.loc[i, j] = q1.home_vs[(i, j)]
        away_at_matrix.loc[i, j] = q1.away_vs[(i, j)]

home_vs_matrix, away_at_matrix

(                       Atlanta Hawks  Boston Celtics  Brooklyn Nets  \
 Atlanta Hawks                      0               1              0   
 Boston Celtics                     0               0              0   
 Brooklyn Nets                      1               1              0   
 Chicago Bulls                      1               1              1   
 Cleveland Cavaliers                1               1              1   
 Dallas Mavericks                   0               0              0   
 Denver Nuggets                     1               1              1   
 Golden State Warriors              0               1              0   
 Houston Rockets                    1               1              1   
 Los Angeles Lakers                 0               1              1   
 Miami Heat                         0               0              0   
 Milwaukee Bucks                    0               0              1   
 New York Knicks                    0               1           

In [17]:
# Home/away date lists as DataFrames

home_dates_df = pd.DataFrame(
    [{"team": t, "date": d} for t in q1.teams for d in q1.home_dates[t]]
)
home_dates_df["date_dt"] = pd.to_datetime(home_dates_df["date"], format="%a, %b %d, %Y")
home_dates_df = home_dates_df.sort_values(["team", "date_dt"]).drop(columns=["date_dt"]).reset_index(drop=True)

away_dates_df = pd.DataFrame(
    [{"team": t, "date": d} for t in q1.teams for d in q1.away_dates[t]]
)
away_dates_df["date_dt"] = pd.to_datetime(away_dates_df["date"], format="%a, %b %d, %Y")
away_dates_df = away_dates_df.sort_values(["team", "date_dt"]).drop(columns=["date_dt"]).reset_index(drop=True)

home_dates_df.head(10), away_dates_df.head(10)

(            team               date
 0  Atlanta Hawks  Mon, Nov 03, 2025
 1  Atlanta Hawks  Fri, Nov 07, 2025
 2  Atlanta Hawks  Sat, Nov 15, 2025
 3  Atlanta Hawks  Mon, Nov 17, 2025
 4  Atlanta Hawks  Wed, Nov 19, 2025
 5  Atlanta Hawks  Sun, Nov 23, 2025
 6  Atlanta Hawks  Thu, Nov 27, 2025
 7  Atlanta Hawks  Fri, Nov 28, 2025
 8  Atlanta Hawks  Sat, Nov 29, 2025
 9  Atlanta Hawks  Thu, Dec 25, 2025,
              team               date
 0   Atlanta Hawks  Sat, Nov 01, 2025
 1   Atlanta Hawks  Wed, Nov 05, 2025
 2   Atlanta Hawks  Tue, Nov 11, 2025
 3   Atlanta Hawks  Thu, Nov 13, 2025
 4   Atlanta Hawks  Fri, Nov 21, 2025
 5   Atlanta Hawks  Mon, Dec 01, 2025
 6  Boston Celtics  Mon, Nov 03, 2025
 7  Boston Celtics  Wed, Nov 05, 2025
 8  Boston Celtics  Tue, Nov 11, 2025
 9  Boston Celtics  Sat, Nov 15, 2025)

In [18]:
# Q1 (a) and (d): chronological display
# Reuse the chronologically sorted DataFrames created above.

home_dates_df.head(10), away_dates_df.head(10)

(            team               date
 0  Atlanta Hawks  Mon, Nov 03, 2025
 1  Atlanta Hawks  Fri, Nov 07, 2025
 2  Atlanta Hawks  Sat, Nov 15, 2025
 3  Atlanta Hawks  Mon, Nov 17, 2025
 4  Atlanta Hawks  Wed, Nov 19, 2025
 5  Atlanta Hawks  Sun, Nov 23, 2025
 6  Atlanta Hawks  Thu, Nov 27, 2025
 7  Atlanta Hawks  Fri, Nov 28, 2025
 8  Atlanta Hawks  Sat, Nov 29, 2025
 9  Atlanta Hawks  Thu, Dec 25, 2025,
              team               date
 0   Atlanta Hawks  Sat, Nov 01, 2025
 1   Atlanta Hawks  Wed, Nov 05, 2025
 2   Atlanta Hawks  Tue, Nov 11, 2025
 3   Atlanta Hawks  Thu, Nov 13, 2025
 4   Atlanta Hawks  Fri, Nov 21, 2025
 5   Atlanta Hawks  Mon, Dec 01, 2025
 6  Boston Celtics  Mon, Nov 03, 2025
 7  Boston Celtics  Wed, Nov 05, 2025
 8  Boston Celtics  Tue, Nov 11, 2025
 9  Boston Celtics  Sat, Nov 15, 2025)

In [19]:
# Q1 (b) and (c): home-vs and away-at opponent count matrices (wide tables)

teams = q1.teams

home_vs_matrix = pd.DataFrame(0, index=teams, columns=teams, dtype=int)
away_at_matrix = pd.DataFrame(0, index=teams, columns=teams, dtype=int)

for i in teams:
    for j in teams:
        if i == j:
            continue
        home_vs_matrix.loc[i, j] = q1.home_vs[(i, j)]
        away_at_matrix.loc[i, j] = q1.away_vs[(i, j)]

home_vs_matrix, away_at_matrix

(                       Atlanta Hawks  Boston Celtics  Brooklyn Nets  \
 Atlanta Hawks                      0               1              0   
 Boston Celtics                     0               0              0   
 Brooklyn Nets                      1               1              0   
 Chicago Bulls                      1               1              1   
 Cleveland Cavaliers                1               1              1   
 Dallas Mavericks                   0               0              0   
 Denver Nuggets                     1               1              1   
 Golden State Warriors              0               1              0   
 Houston Rockets                    1               1              1   
 Los Angeles Lakers                 0               1              1   
 Miami Heat                         0               0              0   
 Milwaukee Bucks                    0               0              1   
 New York Knicks                    0               1           

In [20]:
# Optional but recommended: export Q1 tables to CSV (helpful for report/submission)
export_q1_outputs(q1, OUT_DIR)
OUT_DIR

PosixPath('/Users/luomengzhou/Documents/Opt/Opt Models/Assignment/Project_2/outputs_q1_q2')

In [21]:
for team in q1.teams:
    print(f"\n{'='*60}")
    print(f"TEAM: {team}")
    print(f"{'='*60}")

    # (a) Home dates
    print("\n(a) Home dates:")
    print(q1.home_dates[team])

    # (b) Home matchup counts
    print("\n(b) Home games vs each opponent:")
    for opp in q1.teams:
        if opp != team:
            print(f"{team} vs {opp}: {q1.home_vs[(team, opp)]}")

    # (c) Away matchup counts
    print("\n(c) Away games vs each opponent:")
    for opp in q1.teams:
        if opp != team:
            print(f"{team} at {opp}: {q1.away_vs[(team, opp)]}")

    # (d) Away dates
    print("\n(d) Away dates:")
    print(q1.away_dates[team])


TEAM: Atlanta Hawks

(a) Home dates:
['Mon, Nov 03, 2025', 'Fri, Nov 07, 2025', 'Sat, Nov 15, 2025', 'Mon, Nov 17, 2025', 'Wed, Nov 19, 2025', 'Sun, Nov 23, 2025', 'Thu, Nov 27, 2025', 'Fri, Nov 28, 2025', 'Sat, Nov 29, 2025', 'Thu, Dec 25, 2025']

(b) Home games vs each opponent:
Atlanta Hawks vs Boston Celtics: 1
Atlanta Hawks vs Brooklyn Nets: 0
Atlanta Hawks vs Chicago Bulls: 1
Atlanta Hawks vs Cleveland Cavaliers: 0
Atlanta Hawks vs Dallas Mavericks: 1
Atlanta Hawks vs Denver Nuggets: 0
Atlanta Hawks vs Golden State Warriors: 1
Atlanta Hawks vs Houston Rockets: 0
Atlanta Hawks vs Los Angeles Lakers: 1
Atlanta Hawks vs Miami Heat: 1
Atlanta Hawks vs Milwaukee Bucks: 1
Atlanta Hawks vs New York Knicks: 1
Atlanta Hawks vs Philadelphia 76ers: 0
Atlanta Hawks vs Phoenix Suns: 1
Atlanta Hawks vs Toronto Raptors: 1

(c) Away games vs each opponent:
Atlanta Hawks at Boston Celtics: 0
Atlanta Hawks at Brooklyn Nets: 1
Atlanta Hawks at Chicago Bulls: 1
Atlanta Hawks at Cleveland Cavaliers:

## Q2 — Feasibility Integer Program

Decision variable (binary):

- \(x_{i,j,d} = 1\) if on date \(d\), team \(i\) plays **home** vs team \(j\); else 0.

Constraints enforce:
- each team’s **home dates** match Q1(a)
- each team’s **away dates** match Q1(d)
- **home-vs** opponent totals match Q1(b)
- **away-at** opponent totals match Q1(c)

In [22]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd

In [23]:
def build_q2_model(q1):
    m = gp.Model("Q2_NBA_schedule")

    teams = q1.teams
    dates = q1.dates

    # Decision variable: x[i,j,d] = 1 if team i hosts team j on date d
    x = m.addVars(
        teams, teams, dates,
        vtype=GRB.BINARY,
        name="x"
    )


    for i in teams:
        for d in dates:
            m.addConstr(x[i, i, d] == 0, name=f"no_self_{i}_{d}")


    for i in teams:
        for d in dates:
            m.addConstr(
                gp.quicksum(x[i, j, d] for j in teams if j != i) == q1.home_on[(i, d)],
                name=f"home_date_{i}_{d}"
            )


    for i in teams:
        for d in dates:
            m.addConstr(
                gp.quicksum(x[j, i, d] for j in teams if j != i) == q1.away_on[(i, d)],
                name=f"away_date_{i}_{d}"
            )


    for i in teams:
        for j in teams:
            if i == j:
                continue
            m.addConstr(
                gp.quicksum(x[i, j, d] for d in dates) == q1.home_vs[(i, j)],
                name=f"host_count_{i}_{j}"
            )


    for i in teams:
        for j in teams:
            if i == j:
                continue
            m.addConstr(
                gp.quicksum(x[j, i, d] for d in dates) == q1.away_vs[(i, j)],
                name=f"away_count_{i}_{j}"
            )

    m.setObjective(0, GRB.MINIMIZE)

    return m, x

In [24]:
m, x = build_q2_model(q1)
m.optimize()

print("Status code:", m.Status)
if m.Status == GRB.OPTIMAL:
    print("Feasible Q2 schedule found.")
elif m.Status == GRB.INFEASIBLE:
    print("Model is infeasible.")
else:
    print("Solver ended with status:", m.Status)

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 24.6.0 24G624)

CPU model: Apple M4 Pro
Thread count: 14 physical cores, 14 logical processors, using up to 14 threads



GurobiError: Model too large for size-limited license; visit https://gurobi.com/unrestricted for more information

In [ ]:
def extract_schedule_from_x(q1, x):
    rows = []
    for d in q1.dates:
        for i in q1.teams:
            for j in q1.teams:
                if i == j:
                    continue
                if x[i, j, d].X > 0.5:
                    rows.append({
                        "Date": d,
                        "Home": i,
                        "Visitor": j
                    })
    schedule_df = pd.DataFrame(rows).sort_values(["Date", "Home", "Visitor"]).reset_index(drop=True)
    return schedule_df

In [ ]:
if m.Status == GRB.OPTIMAL:
    schedule_df = extract_schedule_from_x(q1, x)
    display(schedule_df.head(20))
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    schedule_df.to_csv(OUT_DIR / "q2_feasible_schedule.csv", index=False)

,Date,Home,Visitor
0,"Fri, Nov 07, 2025",Atlanta Hawks,Phoenix Suns
1,"Fri, Nov 07, 2025",Boston Celtics,Golden State Warriors
2,"Fri, Nov 07, 2025",Cleveland Cavaliers,Chicago Bulls
3,"Fri, Nov 07, 2025",Dallas Mavericks,Los Angeles Lakers
4,"Fri, Nov 07, 2025",Denver Nuggets,Houston Rockets
5,"Fri, Nov 07, 2025",Milwaukee Bucks,Brooklyn Nets
6,"Fri, Nov 07, 2025",Philadelphia 76ers,New York Knicks
7,"Fri, Nov 07, 2025",Toronto Raptors,Miami Heat
8,"Fri, Nov 21, 2025",Brooklyn Nets,Golden State Warriors
9,"Fri, Nov 21, 2025",Chicago Bulls,Boston Celtics


# Q3 - Schedule Computation

In [ ]:
# Time zone mapping for NBA teams: Eastern=0, Central=1, Mountain=2, Pacific=3
team_tz = {
    "Atlanta Hawks": 0,
    "Boston Celtics": 0,
    "Brooklyn Nets": 0,
    "Charlotte Hornets": 0,
    "Cleveland Cavaliers": 0,
    "Detroit Pistons": 0,
    "Indiana Pacers": 0,
    "Miami Heat": 0,
    "Milwaukee Bucks": 1,
    "New York Knicks": 0,
    "Orlando Magic": 0,
    "Philadelphia 76ers": 0,
    "Toronto Raptors": 0,
    "Washington Wizards": 0,

    "Chicago Bulls": 1,
    "Dallas Mavericks": 1,
    "Houston Rockets": 1,
    "Memphis Grizzlies": 1,
    "Minnesota Timberwolves": 1,
    "New Orleans Pelicans": 1,
    "Oklahoma City Thunder": 1,
    "San Antonio Spurs": 1,

    "Denver Nuggets": 2,
    "Utah Jazz": 2,
    "Phoenix Suns": 2,

    "Golden State Warriors": 3,
    "Los Angeles Clippers": 3,
    "Los Angeles Lakers": 3,
    "Portland Trail Blazers": 3,
    "Sacramento Kings": 3,
}

In [ ]:
def game_dates_for_team(q1, team):
    return sorted(
        set(q1.home_dates[team]) | set(q1.away_dates[team]),
        key=lambda x: pd.to_datetime(x)
    )


def play_at_location_expr(x, teams, t, loc, d):
    if loc == t:
        return gp.quicksum(x[t, j, d] for j in teams if j != t)
    else:
        return x[loc, t, d]

In [ ]:
# time-zone constraints
def add_q3_timezone_constraints(m, x, q1, team_tz):
    teams = q1.teams

    for t in teams:
        dates_t = game_dates_for_team(q1, t)

        # Look at every triple of consecutive games
        for k in range(len(dates_t) - 2):
            d1, d2, d3 = dates_t[k], dates_t[k + 1], dates_t[k + 2]

            # Try every possible location triple (a, b, c)
            for a in teams:
                for b in teams:
                    for c in teams:
                        jump12 = abs(team_tz[a] - team_tz[b])
                        jump23 = abs(team_tz[b] - team_tz[c])

                        # Forbidden if sum >= 4
                        if jump12 + jump23 >= 4:
                            expr1 = play_at_location_expr(x, teams, t, a, d1)
                            expr2 = play_at_location_expr(x, teams, t, b, d2)
                            expr3 = play_at_location_expr(x, teams, t, c, d3)

                            # Cannot have all 3 happen together
                            m.addConstr(
                                expr1 + expr2 + expr3 <= 2,
                                name=f"tz3_{t}_{k}_{a}_{b}_{c}"
                            )


# Q3 model 
def build_q3_model(q1, team_tz):
    m, x = build_q2_model(q1)

    # Add  new Q3 constraints
    add_q3_timezone_constraints(m, x, q1, team_tz)

    return m, x

In [ ]:
m3, x3 = build_q3_model(q1, team_tz)
m3.optimize()

print("Status code:", m3.Status)

if m3.Status == GRB.OPTIMAL:
    print("Q3 feasible schedule found.")
elif m3.Status == GRB.INFEASIBLE:
    print("Q3 model is infeasible.")
    m3.computeIIS()
    m3.write("q3_model.ilp")
    print("IIS written to q3_model.ilp")
else:
    print("Solver ended with status:", m3.Status)


# Extract solved schedule if feasible
def extract_schedule_from_x(q1, x):
    rows = []
    for d in q1.dates:
        for i in q1.teams:
            for j in q1.teams:
                if i != j and x[i, j, d].X > 0.5:
                    rows.append({
                        "Date": d,
                        "Home": i,
                        "Visitor": j
                    })
    return pd.DataFrame(rows).sort_values(["Date", "Home", "Visitor"]).reset_index(drop=True)


if m3.Status == GRB.OPTIMAL:
    q3_schedule_df = extract_schedule_from_x(q1, x3)
    display(q3_schedule_df.head(20))
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    q3_schedule_df.to_csv(OUT_DIR / "q3_schedule.csv", index=False)
    print(f"Saved {(OUT_DIR / 'q3_schedule.csv').as_posix()}")

Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (mac64[x86] - Darwin 24.6.0 24G517)

CPU model: Intel(R) Core(TM) i7-8557U CPU @ 1.70GHz
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 169696 rows, 4096 columns and 963136 nonzeros
Model fingerprint: 0xb1a98ddd
Variable types: 0 continuous, 4096 integer (4096 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [0e+00, 0e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+00]
Presolve removed 156835 rows and 3659 columns
Presolve time: 0.21s
Presolved: 12861 rows, 437 columns, 55324 nonzeros
Variable types: 0 continuous, 437 integer (437 binary)
Performing another presolve...
Presolve removed 11885 rows and 75 columns
Presolve time: 0.01s

Explored 1 nodes (0 simplex iterations) in 0.34 seconds (0.30 work units)
Thread count was 8 (of 8 available processors)

Solution count 0

Model is infeasible
Best objective -, best bound -, gap -
Sta

## Model was infeasible because:
This model falls into an infeasible situation in response to the inclusion of an extra constraint in Question 3. In Question 2, the current model already requires all teams to play based on a fixed home/away schedule and have to meet every other team on a predetermined number of occasions. These two sets of constraints completely dictate when each team is going to play and how often they would meet each other.

In Question 3, one more constraint is added: teams cannot play three games in sequence where the sum of their time zone difference exceeds four. It is another limitation on game orders and sequences of games played by teams. The reason for infeasibility is that these requirements could be mutually exclusive. Because the dates and games are predetermined according to Question 2, the model lacks the freedom of scheduling games or varying the number of meetings between teams. As a result, it leads some teams to have certain games in a certain sequence, for example, from Pacific to Eastern and then to Mountain zones, violating the time zone requirement.

The Irreducible Inconsistent Subsystem (IIS), produced by Gurobi, proves this conflict. The IIS represents a set of constraints (only 18 constraints) that make the whole system infeasible. They consist of some date constraints for home/away teams, constraints regarding the number of match-ups, and new time zone constraints, confirming that this combination of conditions makes the whole problem infeasible.

Overall, the model is infeasible because the combination of fixed schedule constraints of Question 2 and a new travel constraint of Question 3 results in no feasible solution.

## Q4 - Improvement to Current Schedule

In [ ]:
import math

#time zone mapping
team_tz = {
    "Atlanta Hawks": 0,
    "Boston Celtics": 0,
    "Brooklyn Nets": 0,
    "Charlotte Hornets": 0,
    "Cleveland Cavaliers": 0,
    "Detroit Pistons": 0,
    "Indiana Pacers": 0,
    "Miami Heat": 0,
    "Milwaukee Bucks": 1,
    "New York Knicks": 0,
    "Orlando Magic": 0,
    "Philadelphia 76ers": 0,
    "Toronto Raptors": 0,
    "Washington Wizards": 0,

    "Chicago Bulls": 1,
    "Dallas Mavericks": 1,
    "Houston Rockets": 1,
    "Memphis Grizzlies": 1,
    "Minnesota Timberwolves": 1,
    "New Orleans Pelicans": 1,
    "Oklahoma City Thunder": 1,
    "San Antonio Spurs": 1,

    "Denver Nuggets": 2,
    "Utah Jazz": 2,
    "Phoenix Suns": 2,

    "Golden State Warriors": 3,
    "Los Angeles Clippers": 3,
    "Los Angeles Lakers": 3,
    "Portland Trail Blazers": 3,
    "Sacramento Kings": 3,
}

In [ ]:
def game_dates_for_team(q1, team):
    return sorted(
        set(q1.home_dates[team]) | set(q1.away_dates[team]),
        key=lambda x: pd.to_datetime(x)
    )

def play_at_location_expr(x, teams, t, loc, d):
    # loc == t means team t is home on d
    # loc != t means team t is away at loc on d
    if loc == t:
        return gp.quicksum(x[t, j, d] for j in teams if j != t)
    else:
        return x[loc, t, d]

In [ ]:
def build_q4_model(q1, team_tz):
    # Start from Q2 feasible schedule model
    m, x = build_q2_model(q1)

    teams = q1.teams

    z = {}

    team_travel = m.addVars(teams, vtype=GRB.CONTINUOUS, lb=0, name="team_travel")

    T = m.addVar(vtype=GRB.CONTINUOUS, lb=0, name="max_team_travel")

    for t in teams:
        dates_t = game_dates_for_team(q1, t)
        pair_cost_terms = []

        for k in range(len(dates_t) - 1):
            d1 = dates_t[k]
            d2 = dates_t[k + 1]

            for a in teams:
                for b in teams:
                    z[t, k, a, b] = m.addVar(vtype=GRB.BINARY, name=f"z_{t}_{k}_{a}_{b}")

            # Exactly one location pair must be chosen for this consecutive pair
            m.addConstr(
                gp.quicksum(z[t, k, a, b] for a in teams for b in teams) == 1,
                name=f"one_pair_{t}_{k}"
            )

            # Link first location
            for a in teams:
                m.addConstr(
                    gp.quicksum(z[t, k, a, b] for b in teams) == play_at_location_expr(x, teams, t, a, d1),
                    name=f"link_first_{t}_{k}_{a}"
                )

            # Link second location
            for b in teams:
                m.addConstr(
                    gp.quicksum(z[t, k, a, b] for a in teams) == play_at_location_expr(x, teams, t, b, d2),
                    name=f"link_second_{t}_{k}_{b}"
                )

            # Add travel cost for this consecutive pair
            pair_cost_terms.append(
                gp.quicksum(abs(team_tz[a] - team_tz[b]) * z[t, k, a, b] for a in teams for b in teams)
            )

        # Define total travel for team t
        if len(pair_cost_terms) > 0:
            m.addConstr(team_travel[t] == gp.quicksum(pair_cost_terms), name=f"travel_sum_{t}")
        else:
            m.addConstr(team_travel[t] == 0, name=f"travel_sum_{t}")

        # T is the max over all teams
        m.addConstr(T >= team_travel[t], name=f"max_bound_{t}")

    # Objective: minimize maximum travel across teams
    m.setObjective(T, GRB.MINIMIZE)

    return m, x, z, team_travel, T

In [ ]:
m4, x4, z4, team_travel4, T4 = build_q4_model(q1, team_tz)
m4.optimize()

print("Status code:", m4.Status)

if m4.Status == GRB.OPTIMAL:
    print("Q4 optimal schedule found.")
    print("Minimum possible maximum team travel =", T4.X)
elif m4.Status == GRB.INFEASIBLE:
    print("Q4 model is infeasible.")
else:
    print("Solver ended with status:", m4.Status)

Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (mac64[x86] - Darwin 24.6.0 24G517)

CPU model: Intel(R) Core(TM) i7-8557U CPU @ 1.70GHz
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 9200 rows, 65553 columns and 254704 nonzeros
Model fingerprint: 0xf63caa0f
Variable types: 17 continuous, 65536 integer (65536 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]
Presolve removed 7948 rows and 64219 columns
Presolve time: 0.04s
Presolved: 1252 rows, 1334 columns, 7447 nonzeros
Variable types: 0 continuous, 1334 integer (1317 binary)
Found heuristic solution: objective 25.0000000
Found heuristic solution: objective 21.0000000
Performing another presolve...
Presolve removed 662 rows and 561 columns
Presolve time: 0.04s

Explored 1 nodes (0 simplex iterations) in 0.14 seconds (0.11 work units)
Thread count was 8 (of 8 availab

In [ ]:
def extract_schedule_from_x(q1, x):
    rows = []
    for d in q1.dates:
        for i in q1.teams:
            for j in q1.teams:
                if i != j and x[i, j, d].X > 0.5:
                    rows.append({
                        "Date": d,
                        "Home": i,
                        "Visitor": j
                    })
    return pd.DataFrame(rows).sort_values(["Date", "Home", "Visitor"]).reset_index(drop=True)

In [ ]:
# download optimal schedule
if m4.Status == GRB.OPTIMAL:
    q4_schedule_df = extract_schedule_from_x(q1, x4)
    display(q4_schedule_df.head(20))
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    q4_schedule_df.to_csv(OUT_DIR / "q4_schedule.csv", index=False)
    print(f"Saved {(OUT_DIR / 'q4_schedule.csv').as_posix()}")

,Date,Home,Visitor
0,"Fri, Nov 07, 2025",Atlanta Hawks,Los Angeles Lakers
1,"Fri, Nov 07, 2025",Boston Celtics,Phoenix Suns
2,"Fri, Nov 07, 2025",Cleveland Cavaliers,New York Knicks
3,"Fri, Nov 07, 2025",Dallas Mavericks,Golden State Warriors
4,"Fri, Nov 07, 2025",Denver Nuggets,Houston Rockets
5,"Fri, Nov 07, 2025",Milwaukee Bucks,Brooklyn Nets
6,"Fri, Nov 07, 2025",Philadelphia 76ers,Chicago Bulls
7,"Fri, Nov 07, 2025",Toronto Raptors,Miami Heat
8,"Fri, Nov 21, 2025",Brooklyn Nets,Golden State Warriors
9,"Fri, Nov 21, 2025",Chicago Bulls,Boston Celtics


Saved q4_schedule.csv


In [ ]:
if m4.Status == GRB.OPTIMAL:
    q4_travel_df = pd.DataFrame({
        "Team": q1.teams,
        "Optimized_Total_TZ_Travel": [team_travel4[t].X for t in q1.teams]
    }).sort_values("Optimized_Total_TZ_Travel", ascending=False).reset_index(drop=True)

    display(q4_travel_df)

,Team,Optimized_Total_TZ_Travel
0,Golden State Warriors,21.0
1,Los Angeles Lakers,21.0
2,Boston Celtics,19.0
3,Brooklyn Nets,16.0
4,Cleveland Cavaliers,16.0
5,New York Knicks,15.0
6,Denver Nuggets,12.0
7,Philadelphia 76ers,12.0
8,Phoenix Suns,11.0
9,Toronto Raptors,11.0


In [ ]:
# Compute travel in the current schedule for comparison

def compute_current_schedule_tz_travel(df, team_tz):
    df = df.copy()
    df["Date_dt"] = pd.to_datetime(df["Date"], format="%a, %b %d, %Y")
    df = df.sort_values("Date_dt")

    teams = sorted(set(df["Home"]).union(set(df["Visitor"])))
    team_travel = {}

    for t in teams:
        team_games = df[(df["Home"] == t) | (df["Visitor"] == t)].copy()
        team_games = team_games.sort_values("Date_dt")

        locations = []
        for _, row in team_games.iterrows():
            if row["Home"] == t:
                locations.append(t)
            else:
                locations.append(row["Home"])   # away game played at opponent's arena

        total = 0
        for k in range(len(locations) - 1):
            total += abs(team_tz[locations[k]] - team_tz[locations[k + 1]])

        team_travel[t] = total

    return team_travel


current_travel = compute_current_schedule_tz_travel(df, team_tz)

current_travel_df = pd.DataFrame({
    "Team": list(current_travel.keys()),
    "Current_Total_TZ_Travel": list(current_travel.values())
}).sort_values("Current_Total_TZ_Travel", ascending=False).reset_index(drop=True)

display(current_travel_df)

,Team,Current_Total_TZ_Travel
0,Golden State Warriors,25
1,Los Angeles Lakers,19
2,Phoenix Suns,18
3,New York Knicks,15
4,Brooklyn Nets,14
5,Boston Celtics,13
6,Cleveland Cavaliers,13
7,Denver Nuggets,12
8,Philadelphia 76ers,12
9,Dallas Mavericks,11


In [ ]:
# compare current vs optimal
if m4.Status == GRB.OPTIMAL:
    comparison_df = current_travel_df.merge(
        q4_travel_df,
        on="Team",
        how="inner"
    )

    comparison_df["Improvement"] = (
        comparison_df["Current_Total_TZ_Travel"] - comparison_df["Optimized_Total_TZ_Travel"]
    )

    display(comparison_df.sort_values("Improvement", ascending=False).reset_index(drop=True))

    print("Current max team travel:", comparison_df["Current_Total_TZ_Travel"].max())
    print("Optimized max team travel:", comparison_df["Optimized_Total_TZ_Travel"].max())

,Team,Current_Total_TZ_Travel,Optimized_Total_TZ_Travel,Improvement
0,Phoenix Suns,18,11.0,7.0
1,Golden State Warriors,25,21.0,4.0
2,Milwaukee Bucks,9,6.0,3.0
3,Dallas Mavericks,11,9.0,2.0
4,Miami Heat,7,6.0,1.0
5,New York Knicks,15,15.0,0.0
6,Denver Nuggets,12,12.0,0.0
7,Philadelphia 76ers,12,12.0,0.0
8,Toronto Raptors,11,11.0,0.0
9,Houston Rockets,9,9.0,0.0


Current max team travel: 25
Optimized max team travel: 21.0
